In [ ]:
!pip install requests --user

In [ ]:
!python3 cohort_query_gradable_cgm.py

In [ ]:
!python3 diagnose_cohort_query.py

In [ ]:
!python3 cohort_query_gradable_cgm.py

In [ ]:
   !python3 package_for_repository.py

In [ ]:
!echo "--- what's already in active_stage_value_counts.json? ---"
!cat "/home/jupyter-ainaperu/diabetes_registry_project/abstract values/active_stage_value_counts.json"
!echo ""
!echo "--- what's already in stage_value_counts_checkpoint.json? ---"
!cat "/home/jupyter-ainaperu/diabetes_registry_project/abstract values/stage_value_counts_checkpoint.json"
!echo ""
!echo "--- where does the diagnosis import script actually read source data from? ---"
!head -40 "/home/jupyter-ainaperu/diabetes_registry_project/diagnosis/diagnosis_step2_import.py"
!echo ""
!echo "--- where does the retinal import script read its manifest from? ---"
!head -40 "/home/jupyter-ainaperu/diabetes_registry_project/imagin/retinal_step1_metadata.py"

In [ ]:
!echo "--- condition_occurrence.csv, following symlinks this time ---"
!find -L ~ -iname "*condition_occurrence*" 2>/dev/null
!echo ""
!echo "--- what's actually under ~/AI-READI (following the symlink) ---"
!ls ~/AI-READI 2>/dev/null
!echo ""
!echo "--- how full_path is built in hourly_step2_final.py ---"
!grep -n -B 15 "hourly = aggregate_simple\|hourly = aggregate_glucose" "/home/jupyter-ainaperu/diabetes_registry_project/hourly_import/hourly_step2_final.py"
!echo ""
!echo "--- how full_path is built in env_step2_import.py ---"
!grep -n -B 15 "rows = read_env_csv" "/home/jupyter-ainaperu/diabetes_registry_project/envt/env_step2_import.py"

In [ ]:
!python3 stratified_sample_verification.py --probe

In [ ]:
import csv, sys
sys.path.insert(0, "/home/jupyter-ainaperu/diabetes_registry_project/envt")
from env_aggregation_logic_v2 import read_env_csv, aggregate_env_column

AI_READI_FIXED_ROOT = "/home/jupyter-ainaperu/AI-READI-fixed"

with open(f"{AI_READI_FIXED_ROOT}/environment/manifest.tsv") as f:
    env_manifest = {row["person_id"]: row for row in csv.DictReader(f, delimiter="\t")}

row = env_manifest.get("7767", {})
fp_suffix = row.get("env_sensor_filepath")
print("filepath from manifest:", fp_suffix)

full_path = f"{AI_READI_FIXED_ROOT}{fp_suffix}" if fp_suffix.startswith("/") else f"{AI_READI_FIXED_ROOT}/{fp_suffix}"
rows = read_env_csv(full_path)
print("read_env_csv rows:", len(rows))

for column in ["pm1", "pm2.5", "pm4", "pm10", "hum", "temp", "voc"]:
    result = aggregate_env_column(rows, column)
    first_key = next(iter(result), None)
    print(f"\n{column} -> aggregate_env_column() output:")
    print(" key example:", repr(first_key))
    print(" value example:", repr(result.get(first_key)) if first_key else None)

In [ ]:
!python3 stratified_sample_verification.py

In [ ]:
!python3 stratified_sample_verification.py

In [ ]:
import sys, os
sys.path.insert(0, "/home/jupyter-ainaperu/diabetes_registry_project/abs")
import stratified_sample_verification as S

participants = S.get_all_participants_with_person_id()
sampled = S.sample_participants(500, S.RANDOM_SEED, participants)

# ---------- GLUCOSE mismatches ----------
print("=" * 70)
print("GLUCOSE MISMATCHES")
print("=" * 70)
bg = S.load_manifest_tsv(S.BG_MANIFEST)
shown = 0
for tei, pid in sampled:
    if shown >= 15:
        break
    rows = bg.get(pid, [{}])
    fp_suffix = rows[0].get("glucose_filepath")
    if not fp_suffix:
        continue
    try:
        source_hourly = S.aggregate_glucose(S.full_path(fp_suffix))
    except Exception:
        continue
    events = S.get_all_events(tei, S.M.CGM_GLUCOSE_STAGE_UID)
    for event in events:
        occurred = event.get("occurredAt")
        if not occurred:
            continue
        key = S.hour_key(occurred)
        src_match = next((v for k, v in source_hourly.items() if S.hour_key(k) == key), None)
        if src_match is None:
            continue
        for field, de_uid in S.GLUCOSE_FIELD_MAP.items():
            dhis2_val = S.event_data_value(event, de_uid)
            src_val = src_match.get(field)
            if not S.close_enough(dhis2_val, src_val):
                print(f"person={pid} hour={occurred} field={field:12s} dhis2={dhis2_val!r:15} source={src_val!r}")
                shown += 1
    if shown >= 15:
        break

# ---------- ENVIRONMENTAL mismatches ----------
print()
print("=" * 70)
print("ENVIRONMENTAL MISMATCHES")
print("=" * 70)
env = S.load_manifest_tsv(S.ENV_MANIFEST)
shown = 0
for tei, pid in sampled:
    if shown >= 15:
        break
    rows = env.get(pid, [{}])
    fp_suffix = rows[0].get("env_sensor_filepath")
    if not fp_suffix or fp_suffix == "None":
        continue
    try:
        source_rows = S.read_env_csv(S.full_path(fp_suffix))
    except Exception:
        continue
    for column, stage_uid in S.ENV_COLUMN_TO_STAGE.items():
        try:
            source_hourly = S.aggregate_env_column(source_rows, column)
        except Exception:
            continue
        events = S.get_all_events(tei, stage_uid)
        for event in events:
            occurred = event.get("occurredAt")
            if not occurred:
                continue
            key = S.hour_key(occurred)
            src_match = next((v for k, v in source_hourly.items() if S.hour_key(k) == key), None)
            if src_match is None:
                continue
            field_map = dict(S.ENV_FIELD_MAP)
            if column == "hum":
                field_map.update(S.ENV_HUMIDITY_EXTRA_MAP)
            for field, de_uid in field_map.items():
                dhis2_val = S.event_data_value(event, de_uid)
                src_val = src_match.get(field)
                if not S.close_enough(dhis2_val, src_val):
                    print(f"person={pid} col={column:6s} hour={occurred} field={field:14s} dhis2={dhis2_val!r:15} source={src_val!r}")
                    shown += 1
    if shown >= 15:
        break

In [ ]:
import sys
sys.path.insert(0, "/home/jupyter-ainaperu/diabetes_registry_project/abs")
import stratified_sample_verification as S

participants = S.get_all_participants_with_person_id()
sampled = S.sample_participants(500, S.RANDOM_SEED, participants)

# ---------- GLUCOSE mismatches (expect ~1,078) ----------
print("GLUCOSE MISMATCHES")
bg = S.load_manifest_tsv(S.BG_MANIFEST)
shown = 0
for tei, pid in sampled:
    if shown >= 20:
        break
    rows = bg.get(pid, [{}])
    fp_suffix = rows[0].get("glucose_filepath")
    if not fp_suffix:
        continue
    try:
        source_hourly = S.aggregate_glucose(S.full_path(fp_suffix))
    except Exception:
        continue
    events = S.get_all_events(tei, S.M.CGM_GLUCOSE_STAGE_UID)
    for event in events:
        occurred = event.get("occurredAt")
        if not occurred:
            continue
        key = S.hour_key(occurred)
        src_match = next((v for k, v in source_hourly.items() if S.hour_key(k) == key), None)
        if src_match is None:
            continue
        for field, de_uid in S.GLUCOSE_FIELD_MAP.items():
            dhis2_val = S.event_data_value(event, de_uid)
            src_val = src_match.get(field)
            if not S.close_enough(dhis2_val, src_val):
                print(f"  person={pid} field={field:12s} dhis2={dhis2_val!r:12} source={src_val!r}")
                shown += 1

# ---------- ENVIRONMENTAL mismatches (expect ~7) ----------
print("\nENVIRONMENTAL MISMATCHES")
env = S.load_manifest_tsv(S.ENV_MANIFEST)
shown = 0
for tei, pid in sampled:
    if shown >= 20:
        break
    rows = env.get(pid, [{}])
    fp_suffix = rows[0].get("env_sensor_filepath")
    if not fp_suffix or fp_suffix == "None":
        continue
    try:
        source_rows = S.read_env_csv(S.full_path(fp_suffix))
    except Exception:
        continue
    for column, stage_uid in S.ENV_COLUMN_TO_STAGE.items():
        try:
            source_hourly = S.aggregate_env_column(source_rows, column)
        except Exception:
            continue
        events = S.get_all_events(tei, stage_uid)
        for event in events:
            occurred = event.get("occurredAt")
            if not occurred:
                continue
            key = S.hour_key(occurred)
            src_match = next((v for k, v in source_hourly.items() if S.hour_key(k) == key), None)
            if src_match is None:
                continue
            field_map = dict(S.ENV_FIELD_MAP)
            if column == "hum":
                field_map.update(S.ENV_HUMIDITY_EXTRA_MAP)
            for field, de_uid in field_map.items():
                dhis2_val = S.event_data_value(event, de_uid)
                src_val = src_match.get(field)
                if not S.close_enough(dhis2_val, src_val):
                    print(f"  person={pid} col={column:6s} field={field:14s} dhis2={dhis2_val!r:12} source={src_val!r}")
                    shown += 1

In [ ]:
import sys
sys.path.insert(0, "/home/jupyter-ainaperu/diabetes_registry_project/abs")
import stratified_sample_verification as S

bg = S.load_manifest_tsv(S.BG_MANIFEST)
row = bg.get("7315", [{}])[0]
fp_suffix = row.get("glucose_filepath")
full_path = S.full_path(fp_suffix)

print(f"Reading raw source file: {full_path}\n")
import csv
with open(full_path) as f:
    reader = csv.DictReader(f)
    for r in reader:
        ts = r.get("effective_time_frame") or r.get("timestamp") or list(r.values())[0]
        if "2024-06-04T00" in str(ts) or "2024-06-04T01" in str(ts) or "2024-06-04T02" in str(ts):
            print(r)

In [ ]:
import json

path = "/home/jupyter-ainaperu/AI-READI-fixed/wearable_blood_glucose/continuous_glucose_monitoring/dexcom_g6/7315/7315_DEX.json"

with open(path) as f:
    data = json.load(f)

# Figure out the real shape first, rather than assume
print("Top-level type:", type(data))
if isinstance(data, dict):
    print("Top-level keys:", list(data.keys()))
    # try common wrapping patterns
    records = data.get("body") or data.get("data") or data.get("results") or data
else:
    records = data

print("Number of records:", len(records) if hasattr(records, "__len__") else "unknown")
print("\nFirst record, raw:")
print(json.dumps(records[0], indent=2) if isinstance(records, list) else records)

# Now find entries actually inside the hour we care about
print("\n--- Entries near 2024-06-04 00:00 to 03:00 ---")
count_shown = 0
for r in records:
    r_str = json.dumps(r)
    if "2024-06-04T00" in r_str or "2024-06-04T01" in r_str or "2024-06-04T02" in r_str:
        print(json.dumps(r, indent=2))
        count_shown += 1
    if count_shown >= 5:
        break